In [2]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

print("¡Todo funciona correctamente!")

Python: /home/alumne/Escriptori/entorn_python/src/entorn_python/.venv/bin/python
Pandas: 3.1.0
NumPy: 2.5.4
¡Todo funciona correctamente!


In [3]:
import sys
import pandas as pd
import numpy as np

print(sys.executable)
print(pd.__version__)
print(np.__version__)

/home/alumne/Escriptori/entorn_python/src/entorn_python/.venv/bin/python
3.1.0
2.5.4


# Exercici 1
Descarregueu, al vostre directori de treball, els scripts generar_dataset.py i generar_domini.py
(els trobareu al mateix lloc que aquesta pràctica) i executeu el primer amb uv:
$ uv run generar_dataset.py

Comproveu que s’ha creat reserves_brut.csv. L’script fa servir sempre la mateixa llavor alea-
tòria: si l’executeu més d’una vegada, obtindreu exactament el mateix fitxer.

In [ ]:
from pathlib import Path
import io
import json
import urllib.request

import pandas as pd

ADJUNT = "reserves_brut.csv"
candidats = [Path("data") / ADJUNT, Path(ADJUNT)]
path = next((p for p in candidats if p.exists()), None)

if path is None:
    try:
        from google.colab import files  # només existeix dins Colab
    except ImportError:
        raise FileNotFoundError(f"No s'ha trobat {ADJUNT}: cal posar-lo al costat del notebook o a data/.")
    print(f"Cal pujar {ADJUNT} (l'adjunt de Classroom):")
    pujat = files.upload()
    path = Path(next(iter(pujat)))

print("Llegint:", path.resolve())

Cal pujar reserves_brut.csv (l'adjunt de Classroom):


Saving reserves_brut.csv to reserves_brut.csv
Llegint: /content/reserves_brut.csv


In [4]:
df = pd.read_csv("reserves_brut.csv")

In [5]:
print(df.head())

      id client_id    zona    km  import        estat    canal   creada_el  \
0  R0231      C155   PALMA   7.2    0,00  cancel·lada      WEB  2026-05-13   
1  R0135      C107   PALMA  28.3   86,62   confirmada      APP  2026-06-04   
2  R0247      C092  CALVIA  16.0   57,92   confirmada  TELEFON  2026-04-23   
3  R0490      C127   PALMA  30.9  107,32   confirmada      WEB  12/02/2026   
4  R0504      C026  palma   33.8  105,23   confirmada      WEB  2026-04-09   

  recollida_el  passatgers codi_postal                 email  pagat  
0   2026-05-21         3.0       07150  client155@correu.cat  false  
1   2026-06-04         2.0       07150  client107@correu.cat     Sí  
2   2026-04-30         3.0       07010                   NaN     si  
3   2026-02-21         1.0       07001  client127@correu.cat      1  
4   2026-04-27         2.0       07500  client026@correu.cat      1  


# Exercici 2
Partint de les observacions que vau anotar a l’exercici 2 de la UT1, feu una radiografia ràpida de

reserves_brut.csv (dimensions, tipus inferits, cardinalitats i estadístics de les columnes numè-
riques) i comproveu quines es repeteixen en aquest conjunt. Per a cadascun dels quatre pecats,

anoteu com a mínim una sospita que ja us cridi l’atenció. Les comprovareu a les seccions següents.

In [7]:
# df = pd.read_csv(path)
##### La seqüència canònica #####

#Dimensions
print("Dimensions:")
print(df.shape)
print()

#Tipus
print("Tipus:")
print(df.dtypes)
print()

#Mostra
print("Mostra:")
print(df.head())
print()


#Cardinalitats
print("Cardinalitats:")
print(df.nunique().sort_values())
print()


#Estadistiques
print("Estadistiques:")
print(df.describe(include="number").round(2)) # estadístics
print()

print("zona:")
print(df["zona"].value_counts()) # freqüències de zona
print()

print("Nulls:")
print(df.isna().sum())

Dimensions:
(813, 13)

Tipus:
id                  str
client_id           str
zona                str
km              float64
import              str
estat               str
canal               str
creada_el           str
recollida_el        str
passatgers      float64
codi_postal         str
email               str
pagat               str
dtype: object

Mostra:
      id client_id    zona    km  import        estat    canal   creada_el  \
0  R0231      C155   PALMA   7.2    0,00  cancel·lada      WEB  2026-05-13   
1  R0135      C107   PALMA  28.3   86,62   confirmada      APP  2026-06-04   
2  R0247      C092  CALVIA  16.0   57,92   confirmada  TELEFON  2026-04-23   
3  R0490      C127   PALMA  30.9  107,32   confirmada      WEB  12/02/2026   
4  R0504      C026  palma   33.8  105,23   confirmada      WEB  2026-04-09   

  recollida_el  passatgers codi_postal                 email  pagat  
0   2026-05-21         3.0       07150  client155@correu.cat  false  
1   2026-06-04         2.0

# Variable Domini de negoci

- **id:** R seguida de 4 dígits; única per reserva.
- **client_id:** C seguida de 3 dígits.
- **zona:** PALMA, CALVIA, LLEVANT, NORD.
- **km:** entre 1 i 400.
- **import:** entre 0 i 1500 €; una reserva cancel·lada no es cobra (import 0).
- **estat:** confirmada, cancel·lada, pendent.
- **canal:** WEB, TELEFON, APP.
- **creada_el:** data en format `aaaa-mm-dd`, no posterior a la data d’extracció (`2026-10-01`).
- **recollida_el:** data en format `aaaa-mm-dd`, no anterior a `creada_el`.
- **passatgers:** entre 1 i 8.
- **codi_postal:** `07` seguit de 3 dígits.
- **email:** text amb una única `@` i sense espais.
- **pagat:** si, no.

# Exercici 3
Executeu generar_domini.py i obriu el domini_variables.json que genera. Llegiu el codi de
l’script i expliqueu, amb les vostres paraules:
- què fa la funció infereix_domini i com decideix si una columna rep un rang, una llista de valors o res més que un exemple;
    - Proposa un domini per columna, si es numèrica, proposa un rang. Si te 28 valors o meny, la tracta de categoría i si no, la tracta com a texte i et dona un exemple.
- per a què serveix la constant MAX_CATEGORIES;
    - Per definir si la columna es una categoria o no, en funció d'un nombre que li hem definit.
- què fa la funció aplica_correccions amb el diccionari CORRECCIONS_HUMANES.
    - Garenteix que totes les id segueixin el mateix patró de l'expresió regular "R\\d{4}"



# Exercici 4
Compareu el domini inferit amb la taula de regles de negoci. Trobeu com a mínim quatre columnes
on el domini inferit és incorrecte i, per a cadascuna, digueu quin pecat s’hi ha colat. Expliqueu
per què un validador construït sobre aquest domini inferit no trobaria pràcticament cap error.

- Columna Zona: 12 zones diferents en lloc de 4 -> Mala Codificació
- Columna Email: Valors nulls -> Absència de valors
- Columna KM: Valors exagerats -> Valors atípics
- Columna Còdi Postal: Espais en blanc -> Valors incorrectes

# Exercici 5
Completeu CORRECCIONS_HUMANES a generar_domini.py perquè cada columna tingui el domini
de la taula de negoci, i torneu a generar domini_variables.json. Feu servir "valors" per a
les llistes tancades, "rang" per als intervals numèrics i "patro" (una expressió regular) per als
formats de text. Les regles que relacionen dues columnes (per exemple, una cancel·lada no es
cobra) no hi caben: les programareu a l’exercici següent.

# Exercici 6
Programeu un validador declaratiu que llegeixi el vostre domini_variables.json, en derivi les
regles i hi afegeixi tres regles de consistència entre camps: cancel·lades amb import cobrat,
recollides anteriors a la creació i reserves creades després de la data d’extracció. Les primeres
línies de l’esquelet us donen les conversions de tipus que necessiteu (són feina del pecat 3):

Interpreteu el resultat: hi ha regles que salten per un altre pecat diferent del d’un valor incorrecte?
Quines, i per quin?

In [9]:
import json

with open("domini_variables.json", encoding="utf-8") as f:
    domini = json.load(f)

In [11]:
print(domini)

{'id': {'tipus': 'text', 'exemple': 'R0231', 'patro': '^R\\d{4}$', 'unic': True}, 'client_id': {'tipus': 'text', 'exemple': 'C155', 'patro': '^C\\d{3}$'}, 'zona': {'valors': ['PALMA', 'CALVIA', 'LLEVANT', 'NORD']}, 'km': {'rang': [1, 400]}, 'import': {'tipus': 'text', 'exemple': '0,00', 'rang': [0, 1500], 'regla': 'Si estat és cancel·lada, import ha de ser 0'}, 'estat': {'valors': ['confirmada', 'cancel·lada', 'pendent']}, 'canal': {'valors': ['WEB', 'TELEFON', 'APP']}, 'creada_el': {'tipus': 'text', 'exemple': '2026-05-13', 'patro': '^\\d{4}-\\d{2}-\\d{2}$', 'data_maxima': '2026-10-01'}, 'recollida_el': {'tipus': 'text', 'exemple': '2026-05-21', 'patro': '^\\d{4}-\\d{2}-\\d{2}$', 'regla': 'Data igual o posterior a creada_el'}, 'passatgers': {'rang': [1, 8], 'tipus': 'enter'}, 'codi_postal': {'tipus': 'text', 'exemple': '07150', 'patro': '^07\\d{3}$'}, 'email': {'tipus': 'text', 'exemple': 'client155@correu.cat', 'patro': '^[^@\\s]+@[^@\\s]+$'}, 'pagat': {'valors': ['si', 'no']}}


# Exercici 7
Compteu els duplicats exactes (files idèntiques) i els duplicats lògics (mateix client_id, zona, km
i creada_el, però un id diferent) del conjunt. Assegureu-vos que els duplicats lògics que doneu
no inclouen els exactes. Expliqueu què canvia si feu servir keep=False en lloc del valor per defecte
de duplicated, i quin origen probable té cada tipus de duplicat.

In [12]:
duplicats_exacts = df.duplicated()

print("Duplicats exactes:", duplicats_exacts.sum())

Duplicats exactes: 5


In [13]:
columnes = ['client_id', 'zona', 'km', 'creada_el']

duplicats_logics = (
    df.duplicated(subset=columnes, keep='first')
    & ~df.duplicated(keep='first')
)

print("Duplicats lògics:", duplicats_logics.sum())

Duplicats lògics: 8


keep='first' (per defecte)	Manté la primera aparició i marca les següents com a duplicades.

keep='last'	Manté l'última aparició i marca les anteriors.

keep=False	Marca totes les files que formen part d'un grup de duplicats, inclosa la primera.

Duplicats exactes: probablement provenen d'una importació repetida de dades, una càrrega duplicada d'un fitxer o una execució repetida d'un procés d'inserció.

Duplicats lògics: probablement provenen de registres introduïts més d'una vegada per error, d'una sincronització entre sistemes o d'una nova inserció del mateix trajecte amb un identificador diferent.

# Exercici 8
Compteu els NaN de cada columna. Tingueu en compte que, en llegir el CSV, pandas ja converteix
en NaN alguns textos habituals com "N/A" o "NULL". Feu un inventari dels valors de email que
no tenen forma d’adreça electrònica i identifiqueu quins són pseudo-buits. Identifiqueu també,
a les columnes numèriques, quins valors fan de sentinella.

In [14]:
# Comptar els NaN de cada columna
print(df.isna().sum())

id               0
client_id        0
zona             0
km              16
import          24
estat            0
canal            0
creada_el        0
recollida_el     0
passatgers      12
codi_postal     32
email           59
pagat            0
dtype: int64


In [15]:
print((df.isna().mean() * 100).round(2))

id              0.00
client_id       0.00
zona            0.00
km              1.97
import          2.95
estat           0.00
canal           0.00
creada_el       0.00
recollida_el    0.00
passatgers      1.48
codi_postal     3.94
email           7.26
pagat           0.00
dtype: float64


In [16]:
# Eliminar espais abans i després
emails = df['email'].astype('string').str.strip()

# Patró bàsic de correu electrònic
patro = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'

# Emails que no són NaN i no compleixen el patró
emails_incorrectes = df[
    emails.notna() & ~emails.str.match(patro, na=False)
]

print(emails_incorrectes['email'].value_counts(dropna=False))

email
-             3
              3
desconegut    2
no en té      2
Name: count, dtype: int64


In [21]:
print(emails_incorrectes)

        id client_id     zona    km  import        estat    canal   creada_el  \
152  R0798      C039  LLEVANT  11.5    0,00  cancel·lada      APP  2026-04-25   
168  R0641      C115  LLEVANT  38.8  109,49   confirmada  TELEFON  2026-07-25   
251  R0686      C041  LLEVANT  52.7  188,28   confirmada      WEB  2026-09-30   
281  R0651      C127  LLEVANT  24.2    0,00  cancel·lada      APP  2026-08-21   
282  R0178      C043    PALMA   8.1   24,22   confirmada      WEB  2026-08-14   
441  R0035      C161    PALMA   9.7   30,85   confirmada      WEB  2026-04-06   
586  R0611      C071     NORD  12.1    0,00  cancel·lada      WEB  2026-03-26   
610  R0700      C118  Llevant  40.9  151,40      pendent      APP  2026-08-30   
621  R0738      C006  LLEVANT  16.0   49,80   confirmada      WEB  2026-09-01   
757  R0614      C070  LLEVANT  22.7    0,00  cancel·lada      WEB  2026-01-16   

    recollida_el  passatgers codi_postal       email  pagat  
152   2026-05-06         1.0       07200      

In [28]:
pseudo_buits = [
    '',
    ' ',
    '  ',
    '-',
    '--',
    'N/D',
    'no consta',
    'no en té',
    'sense email',
    'desconegut',
    'NULL',
    'NaN'
]

print(df[
    df['email'].astype('string').str.strip().str.lower().isin(pseudo_buits)
]['email'].value_counts())

email
-             3
              3
desconegut    2
no en té      2
Name: count, dtype: int64


In [18]:
# Seleccionar columnes numèriques
columnes_numeriques = df.select_dtypes(include='number').columns

for columna in columnes_numeriques:
    print(f"\nColumna: {columna}")
    print(df[columna].value_counts(dropna=False).sort_index())


Columna: km
km
-65.7       1
-30.4       1
-23.0       1
 1.4        1
 1.5        2
           ..
 157.8      1
 168.6      1
 179.9      1
 9999.0    12
 NaN       16
Name: count, Length: 394, dtype: int64

Columna: passatgers
passatgers
0.0      10
1.0     295
2.0     256
3.0     149
4.0      88
12.0      1
15.0      1
20.0      1
NaN      12
Name: count, dtype: int64


In [25]:
sentinelles = [-1, -999, 0, 9999]

for columna in columnes_numeriques:
    valors = df[columna].value_counts()

    for valor in sentinelles:
        if valor in valors.index:
            print(columna, valor, "→", valors[valor], "aparicions")

km 9999 → 12 aparicions
passatgers 0 → 10 aparicions


# Exercici 9
Completeu la funció absents_reals, que ha de marcar com a absent qualsevol dels tres graus
d’absència:

PSEUDO = {...} # els pseudo-buits que heu trobat (després de treure espais)

def absents_reals(s, sentinelles=()):

m = ... # NaN honestos

m = m | ... # pseudo-buits: passeu a text i traieu els espais

m = m | ... # sentinelles numèriques

return m

Feu-la servir per construir una taula amb, per a cada columna, el % de NaN, de pseudo-buits, de
sentinelles i el % total d’absents reals. Després, calculeu la mitjana de km tal com és i excloent-ne
les sentinelles, i expliqueu la diferència.

In [29]:
PSEUDO = ['', '-', 'desconegut', 'no en té']

In [30]:
def absents_reals(s, sentinelles=()):

    # NaN honestos
    m = s.isna()

    # Pseudo-buits: convertir a texto y eliminar espacios
    m = m | s.astype('string').str.strip().str.lower().isin(PSEUDO)

    # Sentinelles numèriques
    m = m | s.isin(sentinelles)

    return m

In [31]:
sentinelles = [-1, -999, 9999]

In [33]:
resultats = []

for columna in df.columns:

    s = df[columna]

    # Porcentaje de NaN
    nan = s.isna().mean() * 100

    # Porcentaje de pseudo-buits
    pseudo = (
        s.astype('string')
        .str.strip()
        .str.lower()
        .isin(PSEUDO)
        .mean() * 100
    )

    # Porcentaje de sentinelles
    sent = s.isin(sentinelles).mean() * 100

    # Porcentaje total de ausentes reales
    total = absents_reals(s, sentinelles).mean() * 100

    resultats.append({
        'columna': columna,
        'NaN (%)': nan,
        'Pseudo-buits (%)': pseudo,
        'Sentinelles (%)': sent,
        'Total absents (%)': total
    })

taula_absents = pd.DataFrame(resultats)

print(taula_absents.round(2).T)

                    0          1     2     3       4      5      6   \
columna             id  client_id  zona    km  import  estat  canal   
NaN (%)            0.0        0.0   0.0  1.97    2.95    0.0    0.0   
Pseudo-buits (%)   0.0        0.0   0.0   0.0     0.0    0.0    0.0   
Sentinelles (%)    0.0        0.0   0.0  1.48     0.0    0.0    0.0   
Total absents (%)  0.0        0.0   0.0  3.44    2.95    0.0    0.0   

                          7             8           9            10     11  \
columna            creada_el  recollida_el  passatgers  codi_postal  email   
NaN (%)                  0.0           0.0        1.48         3.94   7.26   
Pseudo-buits (%)         0.0           0.0         0.0          0.0   1.23   
Sentinelles (%)          0.0           0.0         0.0          0.0    0.0   
Total absents (%)        0.0           0.0        1.48         3.94   8.49   

                      12  
columna            pagat  
NaN (%)              0.0  
Pseudo-buits (%)     0.

In [34]:
# Media original
media_original = df['km'].mean()

# Media excluyendo los sentinelles
media_sin_sentinelles = df.loc[
    ~df['km'].isin(sentinelles), 'km'
].mean()

print("Media original:", media_original)
print("Media sin sentinelles:", media_sin_sentinelles)

Media original: 174.70376411543288
Media sin sentinelles: 24.523439490445863


# Exercici 10

Calculeu el % d’email absent (real) per canal i el % de km absent per zona. Classifiqueu ca-
da absència com a MCAR o MAR i justifiqueu-ho. A continuació, compteu les reserves amb

passatgers = 0 per canal: el 0 és una sentinella o un error de teclat? Quina evidència us ho fa
pensar?

In [35]:
# Crear una columna auxiliar amb els absents reals
df['email_absent'] = absents_reals(df['email'])

# Calcular el percentatge per canal
percentatge_email = (
    df.groupby('canal')['email_absent']
    .mean() * 100
)

print(percentatge_email.round(2))

canal
APP         2.68
TELEFON    21.22
WEB         3.07
Name: email_absent, dtype: float64


In [36]:
df['km_absent'] = absents_reals(df['km'], sentinelles=[-1, -999, 9999])

percentatge_km = (
    df.groupby('zona')['km_absent']
    .mean() * 100
)

print(percentatge_km.round(2))

zona
CALVIA      4.83
Calvia      0.00
Calvià      7.14
LLEVANT     3.97
Llevant     3.12
NORD        2.15
Nord        0.00
PALMA       4.46
Palma       3.70
llevant     0.00
nord        0.00
palma       2.50
Name: km_absent, dtype: float64


Hi ha dues categories que hem de distingir:
Tipus	Significat	Exemple
MCAR	L'absència no depèn de cap variable ni de cap característica de les dades.	Un error aleatori del sistema que elimina alguns emails.
MAR	L'absència depèn d'una altra variable observada.	Un canal de reserva concret no demana el correu electrònic.

Com podem interpretar els resultats?
- Si el percentatge d'emails absents és molt diferent segons el canal, això és una evidència que l'absència podria ser MAR.
- Si el percentatge de quilòmetres absents varia segons la zona, també podria ser MAR.
- Si no observem diferències significatives entre grups, podria ser compatible amb MCAR, però no ho demostra.

In [37]:
reserves_zero = (
    df[df['passatgers'] == 0]
    .groupby('canal')
    .size()
)

print(reserves_zero)

canal
TELEFON    8
WEB        2
dtype: int64


In [38]:
percentatge_zero = (
    df.assign(zero=df['passatgers'] == 0)
    .groupby('canal')['zero']
    .mean() * 100
)

print(percentatge_zero.round(2))

canal
APP        0.00
TELEFON    3.27
WEB        0.44
Name: zero, dtype: float64


In [39]:
print(df['passatgers'].value_counts(dropna=False).sort_index())

passatgers
0.0      10
1.0     295
2.0     256
3.0     149
4.0      88
12.0      1
15.0      1
20.0      1
NaN      12
Name: count, dtype: int64


In [40]:
print(
    df[df['passatgers'] == 0]['canal']
    .value_counts()
)

canal
TELEFON    8
WEB        2
Name: count, dtype: int64


# Exercici 11
Poseu un exemple plausible, en aquest negoci de trasllats, d’una variable on l’absència podria ser
MNAR. Expliqueu per què aquest patró no es pot demostrar només amb les dades que teniu i
quin risc tindria omplir-ne els forats amb la mitjana.